In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, to_timestamp, unix_timestamp, when

spark = SparkSession.builder.appName("AttendanceAnalysis").getOrCreate()

In [ ]:
df = spark.read.csv("attendance_logs.csv", header=True, inferSchema=True)

In [ ]:
df = df.withColumn("clockin", to_timestamp("clockin"))
df = df.withColumn("clockout", to_timestamp("clockout"))
df = df.withColumn("workhours", (unix_timestamp("clockout") - unix_timestamp("clockin")) / 3600)

In [ ]:
late_logins = df.filter(col("clockin").substr(12, 8) > "09:15:00")
late_logins.show()

In [ ]:
absences = df.filter(col("status") == "Absent")
absences.show()

In [ ]:
dept_summary = df.groupBy("department").agg(
    avg("workhours").alias("avg_workhours"),
    avg("productivityscore").alias("avg_productivity"),
    count(when(col("status") == "Absent", True)).alias("absence_count")
)
dept_summary.show()

In [ ]:
dept_summary.write.mode("overwrite").csv("attendance_issues_by_department", header=True)